In [2]:
# 🚀 DAY 45: LIVE MODEL RETRAINING
print("🎯 Day44 Alpha → Adaptive Learning | p=0.046")
print("Production online learning")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda')

# Day31→44 model + Day42 live ticks
class RegimeLSTM(nn.Module):
    def __init__(self): super().__init__(); self.lstm = nn.LSTM(20,64,2,batch_first=True); self.head = nn.Linear(64,3)
    def forward(self, x): out,_=self.lstm(x); return torch.softmax(self.head(out[:,-1]),-1)

model = RegimeLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Day42 live ticks → Training data (features + labels)
live_ticks = np.random.randn(1000, 20).astype(np.float32)  # Day42 volume/LTP/etc
labels = np.random.randint(0, 3, 1000)  # True regimes

X = torch.from_numpy(live_ticks).to(device)
y = torch.from_numpy(labels).unsqueeze(1).to(device)
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Online retraining (5 epochs)
model.train()
for epoch in range(5):
    total_loss = 0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = nn.CrossEntropyLoss()(pred, batch_y.squeeze())
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}")

# Production inference test
model.eval()
with torch.no_grad():
    test_pred = model(X[:10])
    accuracy = (test_pred.argmax(1) == y[:10].squeeze()).float().mean()
    print(f"\n✅ Live retrain → Test acc: {accuracy:.0%}")

print("\n🎯 DAY 45: Adaptive production!")
print("• GitHub Day45 → 45/90")


🎯 Day44 Alpha → Adaptive Learning | p=0.046
Production online learning


RuntimeError: size mismatch (got input: [3], target: [64])

In [3]:
# 🚀 DAY 45: LIVE MODEL RETRAINING
print("🎯 Day44 Alpha → Adaptive Learning | p=0.046")
print("Production online learning")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class RegimeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=20, hidden_size=64, num_layers=2, batch_first=True)
        self.head = nn.Linear(64, 3)

    def forward(self, x):
        out, _ = self.lstm(x)          # [B, T, 64]
        logits = self.head(out[:, -1]) # [B, 3]
        return logits

model = RegimeLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Simulated live ticks
live_ticks = np.random.randn(1000, 20).astype(np.float32)
labels = np.random.randint(0, 3, 1000).astype(np.int64)

# Add seq_len = 1  → [N, 1, 20]
X = torch.from_numpy(live_ticks).unsqueeze(1).to(device)
y = torch.from_numpy(labels).to(device)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model.train()
for epoch in range(5):
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()

        logits = model(batch_x)                 # [B, 3]
        loss = criterion(logits, batch_y)       # batch_y: [B]

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    print(f"Epoch {epoch+1}: Loss {total_loss/len(loader):.4f} | Acc {correct/total:.2%}")

model.eval()
with torch.no_grad():
    test_logits = model(X[:10])
    test_pred = test_logits.argmax(dim=1)
    accuracy = (test_pred == y[:10]).float().mean()

print(f"\n✅ Live retrain → Test acc: {accuracy:.0%}")
print("\n🎯 DAY 45: Adaptive production!")
print("• GitHub Day45 → 45/90")

🎯 Day44 Alpha → Adaptive Learning | p=0.046
Production online learning
Epoch 1: Loss 1.1054 | Acc 31.20%
Epoch 2: Loss 1.1047 | Acc 31.20%
Epoch 3: Loss 1.1046 | Acc 31.20%
Epoch 4: Loss 1.1036 | Acc 31.20%
Epoch 5: Loss 1.1035 | Acc 31.20%

✅ Live retrain → Test acc: 60%

🎯 DAY 45: Adaptive production!
• GitHub Day45 → 45/90
